## 1. Customers Validation

In [0]:
%sql
SELECT *
FROM capgeminipro.retail_silver.silver_customers
WHERE CustomerID IS NULL;

In [0]:
%sql
SELECT
    CustomerID,
    COUNT(*) AS cnt

FROM capgeminipro.retail_silver.silver_customers

GROUP BY CustomerID

HAVING cnt > 1;

In [0]:
%sql
-- Show which CustomerIDs are duplicated and their details
SELECT 
    CustomerID,
    COUNT(*) AS duplicate_count,
    COLLECT_LIST(CustomerName) AS names,
    COLLECT_LIST(Email) AS emails
FROM capgeminipro.retail_silver.silver_customers
GROUP BY CustomerID
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC;

In [0]:
%sql
-- Check for NULL or empty email addresses
SELECT *
FROM capgeminipro.retail_silver.silver_customers
WHERE Email IS NULL OR TRIM(Email) = '';

## 2. Products Validation

In [0]:
%sql
-- Check for NULL ProductID
SELECT *
FROM capgeminipro.retail_silver.silver_products
WHERE ProductID IS NULL;

In [0]:
%sql
-- Check for duplicate ProductID
SELECT
    ProductID,
    COUNT(*) AS cnt
FROM capgeminipro.retail_silver.silver_products
GROUP BY ProductID
HAVING cnt > 1;

In [0]:
%sql
SELECT *
FROM capgeminipro.retail_silver.silver_products
WHERE UnitPrice <= 0;

In [0]:
%sql
-- Show products with invalid prices
SELECT 
    ProductID,
    ProductName,
    Category,
    UnitPrice
FROM capgeminipro.retail_silver.silver_products
WHERE UnitPrice <= 0
ORDER BY ProductID;

## 3. Stores Validation

In [0]:
%sql
-- Check for NULL StoreID
SELECT *
FROM capgeminipro.retail_silver.silver_stores
WHERE StoreID IS NULL;

In [0]:
%sql
SELECT *
FROM capgeminipro.retail_silver.silver_stores
WHERE Region IS NULL;

In [0]:
%sql
-- Show stores with NULL regions
SELECT 
    StoreID,
    StoreName,
    Region
FROM capgeminipro.retail_silver.silver_stores
WHERE Region IS NULL
ORDER BY StoreID;

In [0]:
%sql
-- Check for duplicate StoreID
SELECT
    StoreID,
    COUNT(*) AS cnt
FROM capgeminipro.retail_silver.silver_stores
GROUP BY StoreID
HAVING cnt > 1;

## 4. Sales Transactions Validation

In [0]:
%sql
-- Check for NULL TransactionID
SELECT *
FROM capgeminipro.retail_silver.silver_sales
WHERE TransactionID IS NULL;

In [0]:
%sql
-- Check for NULL CustomerID
SELECT *
FROM capgeminipro.retail_silver.silver_sales
WHERE CustomerID IS NULL;

In [0]:
%sql
-- Check for NULL ProductID
SELECT *
FROM capgeminipro.retail_silver.silver_sales
WHERE ProductID IS NULL;

In [0]:
%sql
-- Check for NULL StoreID
SELECT *
FROM capgeminipro.retail_silver.silver_sales
WHERE StoreID IS NULL;

In [0]:
%sql
SELECT *
FROM capgeminipro.retail_silver.silver_sales
WHERE Quantity <= 0;

In [0]:
%sql
-- Show sales with invalid quantities
SELECT 
    TransactionID,
    CustomerID,
    ProductID,
    StoreID,
    Quantity,
    TxnDate
FROM capgeminipro.retail_silver.silver_sales
WHERE Quantity <= 0
ORDER BY TransactionID;

In [0]:
%sql
SELECT
    TransactionID,
    COUNT(*) AS cnt

FROM capgeminipro.retail_silver.silver_sales_clean

GROUP BY TransactionID

HAVING cnt > 1;

### Referential Integrity Checks

In [0]:
%sql
-- Check for CustomerID in sales that don't exist in customers
SELECT DISTINCT s.CustomerID
FROM capgeminipro.retail_silver.silver_sales s
LEFT JOIN capgeminipro.retail_silver.silver_customers c
  ON s.CustomerID = c.CustomerID
WHERE c.CustomerID IS NULL
  AND s.CustomerID IS NOT NULL;

In [0]:
%sql
-- Show sales transactions with CustomerIDs that don't exist in customers table
SELECT 
    s.TransactionID,
    s.CustomerID,
    s.ProductID,
    s.StoreID,
    s.Quantity,
    s.TxnDate
FROM capgeminipro.retail_silver.silver_sales s
LEFT JOIN capgeminipro.retail_silver.silver_customers c
  ON s.CustomerID = c.CustomerID
WHERE c.CustomerID IS NULL
  AND s.CustomerID IS NOT NULL
ORDER BY s.TransactionID;

In [0]:
%sql
-- Check for ProductID in sales that don't exist in products
SELECT DISTINCT s.ProductID
FROM capgeminipro.retail_silver.silver_sales s
LEFT JOIN capgeminipro.retail_silver.silver_products p
  ON s.ProductID = p.ProductID
WHERE p.ProductID IS NULL
  AND s.ProductID IS NOT NULL;

In [0]:
%sql
-- Check for StoreID in sales that don't exist in stores
SELECT DISTINCT s.StoreID
FROM capgeminipro.retail_silver.silver_sales s
LEFT JOIN capgeminipro.retail_silver.silver_stores st
  ON s.StoreID = st.StoreID
WHERE st.StoreID IS NULL
  AND s.StoreID IS NOT NULL;

## 5. Validation Summary

Run this cell to get a summary of all validation checks.

In [0]:
%sql
-- Validation Summary: Count of issues by check type
SELECT 'Customers' AS TableName, 'NULL CustomerID' AS CheckType, COUNT(*) AS IssueCount
FROM capgeminipro.retail_silver.silver_customers
WHERE CustomerID IS NULL

UNION ALL

SELECT 'Customers', 'Duplicate CustomerID', COUNT(*)
FROM (
  SELECT CustomerID, COUNT(*) AS cnt
  FROM capgeminipro.retail_silver.silver_customers
  GROUP BY CustomerID
  HAVING cnt > 1
)

UNION ALL

SELECT 'Customers', 'NULL/Empty Email', COUNT(*)
FROM capgeminipro.retail_silver.silver_customers
WHERE Email IS NULL OR TRIM(Email) = ''

UNION ALL

SELECT 'Products', 'NULL ProductID', COUNT(*)
FROM capgeminipro.retail_silver.silver_products
WHERE ProductID IS NULL

UNION ALL

SELECT 'Products', 'Duplicate ProductID', COUNT(*)
FROM (
  SELECT ProductID, COUNT(*) AS cnt
  FROM capgeminipro.retail_silver.silver_products
  GROUP BY ProductID
  HAVING cnt > 1
)

UNION ALL

SELECT 'Products', 'Invalid UnitPrice (<=0)', COUNT(*)
FROM capgeminipro.retail_silver.silver_products
WHERE UnitPrice <= 0

UNION ALL

SELECT 'Stores', 'NULL StoreID', COUNT(*)
FROM capgeminipro.retail_silver.silver_stores
WHERE StoreID IS NULL

UNION ALL

SELECT 'Stores', 'NULL Region', COUNT(*)
FROM capgeminipro.retail_silver.silver_stores
WHERE Region IS NULL

UNION ALL

SELECT 'Stores', 'Duplicate StoreID', COUNT(*)
FROM (
  SELECT StoreID, COUNT(*) AS cnt
  FROM capgeminipro.retail_silver.silver_stores
  GROUP BY StoreID
  HAVING cnt > 1
)

UNION ALL

SELECT 'Sales', 'NULL TransactionID', COUNT(*)
FROM capgeminipro.retail_silver.silver_sales
WHERE TransactionID IS NULL

UNION ALL

SELECT 'Sales', 'NULL CustomerID', COUNT(*)
FROM capgeminipro.retail_silver.silver_sales
WHERE CustomerID IS NULL

UNION ALL

SELECT 'Sales', 'NULL ProductID', COUNT(*)
FROM capgeminipro.retail_silver.silver_sales
WHERE ProductID IS NULL

UNION ALL

SELECT 'Sales', 'NULL StoreID', COUNT(*)
FROM capgeminipro.retail_silver.silver_sales
WHERE StoreID IS NULL

UNION ALL

SELECT 'Sales', 'Invalid Quantity (<=0)', COUNT(*)
FROM capgeminipro.retail_silver.silver_sales
WHERE Quantity <= 0

UNION ALL

SELECT 'Sales', 'Invalid CustomerID (not in customers)', COUNT(DISTINCT s.CustomerID)
FROM capgeminipro.retail_silver.silver_sales s
LEFT JOIN capgeminipro.retail_silver.silver_customers c ON s.CustomerID = c.CustomerID
WHERE c.CustomerID IS NULL AND s.CustomerID IS NOT NULL

UNION ALL

SELECT 'Sales', 'Invalid ProductID (not in products)', COUNT(DISTINCT s.ProductID)
FROM capgeminipro.retail_silver.silver_sales s
LEFT JOIN capgeminipro.retail_silver.silver_products p ON s.ProductID = p.ProductID
WHERE p.ProductID IS NULL AND s.ProductID IS NOT NULL

UNION ALL

SELECT 'Sales', 'Invalid StoreID (not in stores)', COUNT(DISTINCT s.StoreID)
FROM capgeminipro.retail_silver.silver_sales s
LEFT JOIN capgeminipro.retail_silver.silver_stores st ON s.StoreID = st.StoreID
WHERE st.StoreID IS NULL AND s.StoreID IS NOT NULL

ORDER BY TableName, CheckType;